# C1 01 Colab 환경 측정 (T4)
모델 3개가 T4(15GB)에 올라가는지, 속도가 얼마나 나오는지 확인. 이걸로 코퍼스 크기랑 반복 횟수 정함

- 준비: Drive MyDrive/C1/code/ 에 코드 올려 두기
- 런타임: T4 GPU. 다 도는 데 25~40분 정도

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print(sys.version)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/drive/MyDrive/C1/code'
OUT = '/content/drive/MyDrive/C1/results/01_setup'
import os; os.makedirs(OUT, exist_ok=True)
!ls $CODE

Colab 기본 Python이 3.13이라 vLLM 0.6.6이 바로 안 깔림
uv로 Python 3.12 환경(/content/py312)을 따로 만들고 거기에만 설치. 기본 환경은 안 건드려서 재시작 필요 없음
vLLM 0.6.6.post1은 T4에서 AWQ 돌아가는 버전이라 고정. transformers도 4.47.1로 고정

In [ ]:
!pip -q install uv
!uv venv /content/py312 --python 3.12 -q
!uv pip install -q --python /content/py312/bin/python vllm==0.6.6.post1 transformers==4.47.1
!/content/py312/bin/python -c "import sys, vllm, torch; print(sys.version.split()[0], vllm.__version__, torch.__version__, torch.cuda.is_available())"

In [ ]:
%cd $CODE
!python tests/test_c1_core.py

모델마다 프로세스 따로 실행. 하나 실패해도 다음 모델로 넘어가고 에러는 JSON에 남음

In [ ]:
MODELS = [
    'Qwen/Qwen2.5-Coder-7B-Instruct-AWQ',
    'TheBloke/deepseek-coder-6.7B-instruct-AWQ',
    'LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ',
]
for m in MODELS:
    print('=====', m)
    !/content/py312/bin/python src/measure_model.py {m} {OUT}

In [ ]:
import json, glob
rows = []
for f in sorted(glob.glob(f'{OUT}/measure_*.json')):
    r = json.load(open(f, encoding='utf-8'))
    pk = [p for a in r.get('answers', []) for p in a['packages']]
    rows.append({k: r.get(k) for k in ['model','load_ok','load_sec','tok_per_sec_batched','avg_tokens_per_answer','est_hours_200x10']}
                | {'pkgs': len(pk), 'unregistered': [f"{p['eco']}:{p['name']}({p['src']})" for p in pk if p['exists'] is False]})
for r in rows: print(json.dumps(r, ensure_ascii=False))
json.dump(rows, open(f'{OUT}/summary.json','w',encoding='utf-8'), ensure_ascii=False, indent=1)